# Self-RAG Implementation

## Introduction

In this notebook, I am implementing a simple **Self-RAG (Self-Reflective Retrieval-Augmented Generation)** system.

The main idea of Self-RAG is that the system does not retrieve documents for every question. First, it decides whether retrieval is actually needed. If retrieval is required, it searches the knowledge base, checks whether the retrieved information is relevant, and then generates an answer using that information.

After generating the answer, the system also checks whether the answer is properly supported by the retrieved context. If the result is not good enough, the system can try the retrieval and generation process again.

### Main steps in this notebook

1. Create a small knowledge base
2. Split documents into smaller chunks
3. Generate embeddings using Sentence Transformers
4. Store embeddings in FAISS
5. Perform semantic search using FAISS
6. Perform keyword search using BM25
7. Combine both searches using Reciprocal Rank Fusion (RRF)
8. Decide whether retrieval is needed
9. Evaluate the retrieved context
10. Generate an answer using the context
11. Evaluate the generated answer
12. Retry when the result is not satisfactory

### Architecture

```text
                    User Question
                          |
                          v
                Retrieval Decision
                    /          \
                  No            Yes
                  |              |
                  v              v
            Direct Answer    Hybrid Search
                               /      \
                            FAISS     BM25
                               \      /
                                \    /
                                  RRF
                                   |
                                   v
                          Context Evaluation
                              /       \
                           Bad         Good
                            |            |
                          Retry          v
                                      Generate
                                       Answer
                                          |
                                          v
                                  Answer Evaluation
                                      /       \
                                   Bad         Good
                                    |            |
                                  Retry      Final Answer

In [1]:
!pip -q install -U sentence-transformers faiss-cpu google-genai rank-bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 20.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


In [2]:
import os
import re
import json
import numpy as np
import faiss

from getpass import getpass
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from google import genai

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini initialized successfully.")

Enter your Gemini API key: ··········
Gemini initialized successfully.


In [4]:
documents = [

    {
        "id": "doc1",
        "title": "RAG Introduction",
        "text": """
        Retrieval-Augmented Generation, commonly called RAG, combines
        information retrieval with large language models. Instead of relying
        only on the knowledge stored inside an LLM, RAG retrieves relevant
        information from an external knowledge base and provides that
        information to the language model as context.
        """
    },

    {
        "id": "doc2",
        "title": "Vector Databases",
        "text": """
        A vector database stores numerical representations of data called
        embeddings. These embeddings allow the system to perform semantic
        similarity searches. A question can be converted into an embedding
        and compared with document embeddings.
        """
    },

    {
        "id": "doc3",
        "title": "Embeddings",
        "text": """
        Embeddings are numerical vectors that represent the semantic meaning
        of text. Similar pieces of text tend to have embeddings that are close
        to each other in vector space. Embeddings are commonly used for
        semantic search, recommendation systems, clustering and RAG.
        """
    },

    {
        "id": "doc4",
        "title": "RAG Pipeline",
        "text": """
        A typical RAG pipeline consists of document ingestion, text cleaning,
        text splitting, embedding generation, vector storage, retrieval,
        context construction and language model generation.
        """
    },

    {
        "id": "doc5",
        "title": "Document Chunking",
        "text": """
        Chunking divides large documents into smaller pieces before embedding.
        Good chunking is important because very large chunks can contain
        irrelevant information while very small chunks may lose context.
        """
    },

    {
        "id": "doc6",
        "title": "Semantic Search",
        "text": """
        Semantic search retrieves information based on meaning rather than
        relying only on exact keyword matches. Queries and documents are
        converted into embeddings and their similarity is calculated.
        """
    },

    {
        "id": "doc7",
        "title": "BM25 Keyword Search",
        "text": """
        BM25 is a lexical information retrieval algorithm. It ranks documents
        based on query terms and their importance. It is useful for exact
        keywords, technical terminology, product names and error codes.
        """
    },

    {
        "id": "doc8",
        "title": "Reranking",
        "text": """
        Reranking is a second-stage retrieval process. An initial retriever
        retrieves candidate documents and a reranker scores them according
        to their relevance to the query.
        """
    },

    {
        "id": "doc9",
        "title": "Hybrid Search",
        "text": """
        Hybrid search combines semantic vector search and lexical keyword
        search. Vector search is useful for semantic meaning while keyword
        search is useful for exact terms.
        """
    },

    {
        "id": "doc10",
        "title": "Authentication Errors",
        "text": """
        Authentication errors can occur when credentials are invalid,
        authentication tokens expire or authorization policies reject a
        request.

        ERR-401 indicates an authentication failure.

        ERR-403 indicates that the user is authenticated but does not have
        permission to access a resource.

        Access tokens are used to authenticate requests. Refresh tokens are
        used to obtain new access tokens when an access token expires.
        """
    },

    {
        "id": "doc11",
        "title": "Product API",
        "text": """
        The Product API provides endpoints for creating, updating, deleting
        and retrieving product records. Product records contain a product ID,
        name, price, inventory quantity and category. The API uses JSON for
        request and response bodies.
        """
    },

    {
        "id": "doc12",
        "title": "Query Rewriting",
        "text": """
        Query rewriting transforms a user's original question into a clearer
        and more retrieval-friendly query. It can make important concepts
        explicit and use terminology that better matches the knowledge base.
        """
    },

    {
        "id": "doc13",
        "title": "Query Expansion",
        "text": """
        Query expansion generates multiple alternative search queries from
        one user query. The alternatives can use synonyms, related concepts
        or different formulations.
        """
    },

    {
        "id": "doc14",
        "title": "Multi-Query Retrieval",
        "text": """
        Multi-query retrieval uses multiple search queries for one information
        need. Each query can retrieve different documents and the results can
        then be combined.
        """
    },

    {
        "id": "doc15",
        "title": "Query Decomposition",
        "text": """
        Query decomposition breaks a complex question into smaller
        sub-questions. Each sub-question can be answered independently using
        retrieval and the results can then be combined.
        """
    },

    {
        "id": "doc16",
        "title": "Contextual Retrieval",
        "text": """
        Contextual retrieval improves document chunks by adding information
        about where each chunk came from and what it means in the broader
        document.
        """
    },

    {
        "id": "doc17",
        "title": "Reranker Models",
        "text": """
        A cross-encoder reranker takes a query and candidate document together
        and calculates a relevance score. Rerankers are commonly used after
        an initial high-recall retriever.
        """
    },

    {
        "id": "doc18",
        "title": "Self-RAG",
        "text": """
        Self-RAG is a retrieval-augmented generation approach in which the
        language model evaluates whether retrieval is needed, whether
        retrieved evidence is relevant and whether the generated answer is
        adequately supported.
        """
    }
]

print("Number of documents:", len(documents))

Number of documents: 18


In [5]:
def chunk_text(text, chunk_size=70, overlap=15):

    words = text.split()
    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [6]:
chunks = []

for document in documents:

    document_chunks = chunk_text(
        document["text"],
        chunk_size=70,
        overlap=15
    )

    for chunk_number, chunk in enumerate(document_chunks):

        chunks.append({
            "chunk_id": f'{document["id"]}_chunk_{chunk_number}',
            "document_id": document["id"],
            "title": document["title"],
            "chunk_number": chunk_number,
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 18


In [9]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_dimension = embedding_model.get_sentence_embedding_dimension()

print("Embedding dimension:", embedding_dimension)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


/tmp/ipykernel_421/3933845425.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = embedding_model.get_sentence_embedding_dimension()


In [10]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

print("Embeddings created.")
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings created.
Embedding shape: (18, 384)


In [11]:
vector_index = faiss.IndexFlatIP(embedding_dimension)

vector_index.add(embeddings)

In [12]:
def vector_search(query, top_k=10):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = vector_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        result = chunks[index].copy()

        result["vector_score"] = float(score)
        result["vector_rank"] = rank

        results.append(result)

    return results

In [13]:
question = "What are embeddings?"

results = vector_search(question, top_k=5)

for result in results:

    print("-" * 60)
    print("Title:", result["title"])
    print("Score:", round(result["vector_score"], 3))
    print("Text:", result["text"])

------------------------------------------------------------
Title: Embeddings
Score: 0.767
Text: Embeddings are numerical vectors that represent the semantic meaning of text. Similar pieces of text tend to have embeddings that are close to each other in vector space. Embeddings are commonly used for semantic search, recommendation systems, clustering and RAG.
------------------------------------------------------------
Title: Vector Databases
Score: 0.548
Text: A vector database stores numerical representations of data called embeddings. These embeddings allow the system to perform semantic similarity searches. A question can be converted into an embedding and compared with document embeddings.
------------------------------------------------------------
Title: Document Chunking
Score: 0.411
Text: Chunking divides large documents into smaller pieces before embedding. Good chunking is important because very large chunks can contain irrelevant information while very small chunks may los

In [14]:
def tokenize(text):

    return re.findall(
        r"\b\w+\b",
        text.lower()
    )


tokenized_chunks = [
    tokenize(chunk["text"])
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index created.")

BM25 index created.


In [15]:
def bm25_search(query, top_k=10):

    scores = bm25.get_scores(
        tokenize(query)
    )

    indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(indices, start=1):

        result = chunks[index].copy()

        result["bm25_score"] = float(scores[index])
        result["bm25_rank"] = rank

        results.append(result)

    return results

In [16]:
question = "What does ERR-401 mean?"

results = bm25_search(question, top_k=5)

for result in results:

    print("-" * 60)
    print("Title:", result["title"])
    print("BM25 Score:", round(result["bm25_score"], 3))
    print("Text:", result["text"])

------------------------------------------------------------
Title: Authentication Errors
BM25 Score: 6.051
Text: Authentication errors can occur when credentials are invalid, authentication tokens expire or authorization policies reject a request. ERR-401 indicates an authentication failure. ERR-403 indicates that the user is authenticated but does not have permission to access a resource. Access tokens are used to authenticate requests. Refresh tokens are used to obtain new access tokens when an access token expires.
------------------------------------------------------------
Title: Contextual Retrieval
BM25 Score: 2.811
Text: Contextual retrieval improves document chunks by adding information about where each chunk came from and what it means in the broader document.
------------------------------------------------------------
Title: Reranker Models
BM25 Score: 0.0
Text: A cross-encoder reranker takes a query and candidate document together and calculates a relevance score. Reranke

In [17]:
def reciprocal_rank_fusion(result_lists, k=60):

    fused = {}

    for results in result_lists:

        for rank, result in enumerate(results, start=1):

            chunk_id = result["chunk_id"]

            if chunk_id not in fused:

                fused[chunk_id] = {
                    "chunk": result,
                    "rrf_score": 0.0
                }

            fused[chunk_id]["rrf_score"] += 1 / (k + rank)

    ranked = sorted(
        fused.values(),
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    final_results = []

    for item in ranked:

        result = item["chunk"].copy()

        result["rrf_score"] = item["rrf_score"]

        final_results.append(result)

    return final_results

In [18]:
def hybrid_search(query, top_k=8):

    vector_results = vector_search(
        query,
        top_k=15
    )

    bm25_results = bm25_search(
        query,
        top_k=15
    )

    fused_results = reciprocal_rank_fusion(
        [
            vector_results,
            bm25_results
        ]
    )

    return fused_results[:top_k]

In [19]:
question = "What does ERR-401 mean?"

results = hybrid_search(
    question,
    top_k=5
)

for i, result in enumerate(results, start=1):

    print("-" * 60)
    print("Rank:", i)
    print("Title:", result["title"])
    print("RRF Score:", round(result["rrf_score"], 4))
    print("Text:", result["text"])

------------------------------------------------------------
Rank: 1
Title: Authentication Errors
RRF Score: 0.0328
Text: Authentication errors can occur when credentials are invalid, authentication tokens expire or authorization policies reject a request. ERR-401 indicates an authentication failure. ERR-403 indicates that the user is authenticated but does not have permission to access a resource. Access tokens are used to authenticate requests. Refresh tokens are used to obtain new access tokens when an access token expires.
------------------------------------------------------------
Rank: 2
Title: Reranker Models
RRF Score: 0.0315
Text: A cross-encoder reranker takes a query and candidate document together and calculates a relevance score. Rerankers are commonly used after an initial high-recall retriever.
------------------------------------------------------------
Rank: 3
Title: Self-RAG
RRF Score: 0.031
Text: Self-RAG is a retrieval-augmented generation approach in which the lan

In [20]:
def should_retrieve(question):

    prompt = f"""
You are a Self-RAG retrieval decision component.

Decide whether external retrieval is needed to answer the question.

Return ONLY valid JSON in this format:

{{
    "retrieve": true,
    "reason": "short explanation"
}}

Use false for simple conversational questions.

Use true for questions involving:
- technical facts
- the knowledge base
- exact error codes
- APIs
- RAG concepts
- information requiring evidence

QUESTION:
{question}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    try:

        result = json.loads(text)

        return {
            "retrieve": bool(
                result.get("retrieve", True)
            ),
            "reason": result.get("reason", "")
        }

    except Exception:

        return {
            "retrieve": True,
            "reason": "Could not parse decision, so retrieval is enabled."
        }

In [21]:
questions = [
    "What does ERR-401 mean?",
    "What are embeddings?",
    "Hello, how are you?",
    "What is RAG?",
    "Can you explain vector databases?"
]

for question in questions:

    decision = should_retrieve(question)

    print("-" * 60)
    print("Question:", question)
    print("Retrieve:", decision["retrieve"])
    print("Reason:", decision["reason"])

------------------------------------------------------------
Question: What does ERR-401 mean?
Retrieve: True
Reason: This question asks for the meaning of an exact error code, which is a technical fact requiring specific knowledge and likely evidence from a knowledge base.
------------------------------------------------------------
Question: What are embeddings?
Retrieve: True
Reason: The question asks for a technical definition of 'embeddings', which requires specific factual knowledge related to machine learning and NLP concepts.
------------------------------------------------------------
Question: Hello, how are you?
Retrieve: False
Reason: This is a simple conversational greeting that does not require external retrieval.
------------------------------------------------------------
Question: What is RAG?
Retrieve: True
Reason: RAG is a technical concept in AI/NLP that requires a factual and detailed explanation, which benefits from external retrieval to ensure accuracy and comple

In [22]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(results, start=1):

        context_parts.append(
            f"""
SOURCE {i}

Document:
{result["title"]}

Content:
{result["text"]}
"""
        )

    return "\n".join(context_parts)

In [23]:
def evaluate_context(question, context):

    prompt = f"""
You are a Self-RAG context evaluator.

Evaluate whether the retrieved context is useful for answering the question.

Return ONLY valid JSON:

{{
    "relevant": true,
    "score": 0.0,
    "reason": "short explanation"
}}

Scoring:
1.0 = highly relevant
0.7 = mostly relevant
0.4 = partially relevant
0.0 = irrelevant

QUESTION:
{question}

CONTEXT:
{context}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    try:

        result = json.loads(text)

        score = float(
            result.get("score", 0.0)
        )

        score = max(0.0, min(1.0, score))

        return {
            "relevant": bool(
                result.get(
                    "relevant",
                    score >= 0.6
                )
            ),
            "score": score,
            "reason": result.get("reason", "")
        }

    except Exception:

        return {
            "relevant": False,
            "score": 0.0,
            "reason": "Context evaluation failed."
        }

In [24]:
def generate_answer(question, context):

    prompt = f"""
You are a RAG question-answering system.

Answer the question using the retrieved context.

Rules:
1. Use the context as the primary source.
2. Do not invent unsupported facts.
3. If the context is insufficient, say so.
4. Give a clear and concise answer.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

ANSWER:
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text.strip()

In [25]:
def direct_answer(question):

    prompt = f"""
Answer the following question directly.

Question:
{question}

Give a concise answer.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text.strip()

In [26]:
def evaluate_answer(question, answer, context):

    prompt = f"""
You are a Self-RAG answer evaluator.

Evaluate whether the answer:
1. Answers the question.
2. Is supported by the retrieved context.

Return ONLY valid JSON:

{{
    "supported": true,
    "score": 0.0,
    "reason": "short explanation"
}}

Scoring:
1.0 = well supported
0.7 = mostly supported
0.4 = partially supported
0.0 = unsupported

QUESTION:
{question}

CONTEXT:
{context}

ANSWER:
{answer}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()

    try:

        result = json.loads(text)

        score = float(
            result.get("score", 0.0)
        )

        score = max(0.0, min(1.0, score))

        return {
            "supported": bool(
                result.get(
                    "supported",
                    score >= 0.7
                )
            ),
            "score": score,
            "reason": result.get("reason", "")
        }

    except Exception:

        return {
            "supported": False,
            "score": 0.0,
            "reason": "Answer evaluation failed."
        }

In [27]:
def self_rag(
    question,
    top_k=5,
    max_retries=2,
    context_threshold=0.6,
    answer_threshold=0.7
):

    trace = []

    # --------------------------------------------------
    # STEP 1: Decide whether retrieval is needed
    # --------------------------------------------------

    retrieval_decision = should_retrieve(question)

    trace.append({
        "step": "retrieval_decision",
        "result": retrieval_decision
    })

    # --------------------------------------------------
    # STEP 2: Direct answer if retrieval is not needed
    # --------------------------------------------------

    if not retrieval_decision["retrieve"]:

        answer = direct_answer(question)

        trace.append({
            "step": "direct_answer",
            "result": answer
        })

        return {
            "question": question,
            "used_retrieval": False,
            "answer": answer,
            "sources": [],
            "trace": trace
        }

    # --------------------------------------------------
    # STEP 3: Retrieval
    # --------------------------------------------------

    current_query = question

    best_results = []
    best_context = ""
    best_answer = ""

    best_context_score = 0.0
    best_answer_score = 0.0

    # --------------------------------------------------
    # Retry loop
    # --------------------------------------------------

    for attempt in range(max_retries + 1):

        trace.append({
            "step": "retrieval",
            "attempt": attempt + 1,
            "query": current_query
        })

        results = hybrid_search(
            current_query,
            top_k=top_k
        )

        context = build_context(results)

        # --------------------------------------------------
        # STEP 4: Evaluate context
        # --------------------------------------------------

        context_eval = evaluate_context(
            question,
            context
        )

        trace.append({
            "step": "context_evaluation",
            "attempt": attempt + 1,
            "result": context_eval
        })

        # Save best context
        if context_eval["score"] > best_context_score:

            best_context_score = context_eval["score"]

            best_results = results
            best_context = context

        # --------------------------------------------------
        # Bad context -> retry
        # --------------------------------------------------

        if (
            not context_eval["relevant"]
            or
            context_eval["score"] < context_threshold
        ):

            trace.append({
                "step": "context_retry",
                "attempt": attempt + 1,
                "reason": context_eval["reason"]
            })

            current_query = (
                question +
                " Provide information directly relevant to the question."
            )

            continue

        # --------------------------------------------------
        # STEP 5: Generate answer
        # --------------------------------------------------

        answer = generate_answer(
            question,
            context
        )

        trace.append({
            "step": "generation",
            "attempt": attempt + 1,
            "answer": answer
        })

        # --------------------------------------------------
        # STEP 6: Evaluate answer
        # --------------------------------------------------

        answer_eval = evaluate_answer(
            question,
            answer,
            context
        )

        trace.append({
            "step": "answer_evaluation",
            "attempt": attempt + 1,
            "result": answer_eval
        })

        # Save best answer
        if answer_eval["score"] > best_answer_score:

            best_answer_score = answer_eval["score"]

            best_answer = answer
            best_results = results
            best_context = context

        # --------------------------------------------------
        # Good answer -> finish
        # --------------------------------------------------

        if (
            answer_eval["supported"]
            and
            answer_eval["score"] >= answer_threshold
        ):

            trace.append({
                "step": "final_answer",
                "result": "Answer accepted."
            })

            return {
                "question": question,
                "used_retrieval": True,
                "answer": answer,
                "sources": results,
                "context_score": context_eval["score"],
                "answer_score": answer_eval["score"],
                "trace": trace
            }

        # --------------------------------------------------
        # Bad answer -> retry
        # --------------------------------------------------

        trace.append({
            "step": "answer_retry",
            "attempt": attempt + 1,
            "reason": answer_eval["reason"]
        })

        current_query = (
            question +
            " Retrieve more specific evidence."
        )

    # --------------------------------------------------
    # STEP 7: Fallback
    # --------------------------------------------------

    trace.append({
        "step": "fallback",
        "result": "Returning best available answer."
    })

    if not best_answer:

        best_answer = generate_answer(
            question,
            best_context
        )

    return {
        "question": question,
        "used_retrieval": True,
        "answer": best_answer,
        "sources": best_results,
        "context_score": best_context_score,
        "answer_score": best_answer_score,
        "trace": trace
    }

In [28]:
question = "What does ERR-401 mean and how can it happen?"

result = self_rag(
    question,
    top_k=5,
    max_retries=2
)

print("=" * 60)
print("SELF-RAG RESULT")
print("=" * 60)

print("\nQuestion:")
print(result["question"])

print("\nUsed Retrieval:")
print(result["used_retrieval"])

print("\nAnswer:")
print(result["answer"])

if "context_score" in result:
    print("\nContext Score:",
          round(result["context_score"], 3))

if "answer_score" in result:
    print("Answer Score:",
          round(result["answer_score"], 3))

print("\nSources:")

for source in result["sources"]:
    print("-", source["title"])

SELF-RAG RESULT

Question:
What does ERR-401 mean and how can it happen?

Used Retrieval:
True

Answer:
ERR-401 indicates an authentication failure. This error can occur when credentials are invalid, authentication tokens expire, or authorization policies reject a request.

Context Score: 1.0
Answer Score: 1.0

Sources:
- Authentication Errors
- BM25 Keyword Search
- Contextual Retrieval
- Product API
- Reranking


In [29]:
print("=" * 60)
print("SELF-RAG DECISION TRACE")
print("=" * 60)

for step in result["trace"]:

    print("\nSTEP:", step["step"])

    if "attempt" in step:
        print("Attempt:", step["attempt"])

    if "query" in step:
        print("Query:", step["query"])

    if "result" in step:
        print("Result:", step["result"])

    if "reason" in step:
        print("Reason:", step["reason"])

SELF-RAG DECISION TRACE

STEP: retrieval_decision
Result: {'retrieve': True, 'reason': 'The question asks for the meaning of a specific error code (ERR-401) and its causes, which requires factual technical information likely found in a knowledge base or documentation.'}

STEP: retrieval
Attempt: 1
Query: What does ERR-401 mean and how can it happen?

STEP: context_evaluation
Attempt: 1
Result: {'relevant': True, 'score': 1.0, 'reason': 'Source 1 directly explains that ERR-401 indicates an authentication failure and details how such errors can occur (invalid credentials, expired tokens, rejected requests).'}

STEP: generation
Attempt: 1

STEP: answer_evaluation
Attempt: 1
Result: {'supported': True, 'score': 1.0, 'reason': 'The answer accurately states that ERR-401 indicates an authentication failure and lists the causes (invalid credentials, expired authentication tokens, or rejected authorization policies), all of which are directly supported by Source 1.'}

STEP: final_answer
Result:

In [30]:
question = "Hello, how are you?"

result = self_rag(
    question,
    max_retries=1
)

print("=" * 60)
print("NON-RETRIEVAL TEST")
print("=" * 60)

print("\nQuestion:")
print(result["question"])

print("\nUsed Retrieval:")
print(result["used_retrieval"])

print("\nAnswer:")
print(result["answer"])

NON-RETRIEVAL TEST

Question:
Hello, how are you?

Used Retrieval:
False

Answer:
Hello! I'm doing well, thank you for asking.
